# PaCMAP visualization of ModernMolBERT embeddings

In [ ]:
import json
import re
from pathlib import Path

import colorcet as cc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import pacmap
import pandas as pd
import pyarrow.parquet as pq

## Config

In [ ]:
EMBED_DIR = Path("../../outputs/visualize/best_standard")
FIGURE_DIR = Path("../../figures/pacmap")

# Source parquet used when embedding - needed to join extra property columns.
SOURCE_PARQUET = Path("../../data/pretrain/chembl36_selfies/valid.parquet")

# Column from metadata.parquet to color by.
COLOR_COLUMN = "alogp"

# Additional property columns to join from SOURCE_PARQUET.
# Any subset of the list below - they are merged by chembl_id at load time.
EXTRA_COLUMNS = [
    "psa",
    "hba",
    "hbd",
    "qed_weighted",
    "mw_freebase",
    "rtb",
    "aromatic_rings",
    "heavy_atoms",
    "num_ro5_violations",
]

# Display labels for the color legend.
COLUMN_LABELS = {
    "alogp": "ALogP (lipophilicity)",
    "psa": "Polar surface area (Å²)",
    "hba": "H-bond acceptors",
    "hbd": "H-bond donors",
    "qed_weighted": "QED (drug-likeness)",
    "mw_freebase": "Molecular weight (Da)",
    "rtb": "Rotatable bonds",
    "aromatic_rings": "Aromatic rings",
    "heavy_atoms": "Heavy atom count",
    "num_ro5_violations": "Ro5 violations",
    "is_valid": "Valid SELFIES",
}

# Sweep all labeled measures that are available after loading and joining metadata.
MEASURE_COLUMNS = list(COLUMN_LABELS)

# Expanded grid around best setting (n_neighbors=30, MN_ratio=0.0, FP_ratio=12.0)
# Try more neighbors and some finer MN/FP ratios for more separation
N_NEIGHBORS_GRID = [20, 30, 40, 50]
MN_RATIO_GRID = [0.0, 0.25, 0.5]
FP_RATIO_GRID = [8.0, 10.0, 12.0, 15.0]
PACMAP_PARAMETER_GRID = [
    {"n_neighbors": n_neighbors, "MN_ratio": mn_ratio, "FP_ratio": fp_ratio}
    for n_neighbors in N_NEIGHBORS_GRID
    for mn_ratio in MN_RATIO_GRID
    for fp_ratio in FP_RATIO_GRID
]

## Load

In [ ]:
embeddings = np.load(EMBED_DIR / "embeddings.npy")
meta = pd.read_parquet(EMBED_DIR / "metadata.parquet")

print(f"Embeddings: {embeddings.shape}")
print(f"Metadata columns: {meta.columns.tolist()}")

In [ ]:
if EXTRA_COLUMNS and SOURCE_PARQUET.exists():
    source_columns = set(pq.ParquetFile(SOURCE_PARQUET).schema.names)
    cols_to_load = [c for c in EXTRA_COLUMNS if c not in meta.columns and c in source_columns]
    missing_source_cols = [
        c for c in EXTRA_COLUMNS if c not in meta.columns and c not in source_columns
    ]
    if cols_to_load:
        props = pd.read_parquet(SOURCE_PARQUET, columns=["chembl_id"] + cols_to_load)
        meta = meta.merge(props, on="chembl_id", how="left")
        print(f"Joined {cols_to_load} from source parquet.")
    if missing_source_cols:
        print(f"Warning: missing source columns: {missing_source_cols}")
elif EXTRA_COLUMNS:
    print(f"Warning: SOURCE_PARQUET not found at {SOURCE_PARQUET}")

## Run PaCMAP

In [ ]:
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=5,
    MN_ratio=0.0,
    FP_ratio=12.0,
    verbose=True,
)

coords = reducer.fit_transform(embeddings, init="pca")
print(f"PaCMAP output: {coords.shape}")

available_measures = [c for c in MEASURE_COLUMNS if c in meta.columns]
missing_measures = [c for c in MEASURE_COLUMNS if c not in meta.columns]
print(f"Available measures: {available_measures}")
if missing_measures:
    print(f"Missing measures skipped: {missing_measures}")

## Visualize

In [ ]:
def _slug(value):
    text = str(value).replace(".", "p")
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()


# Measures that should use a sequential Colorcet colormap.
SEQUENTIAL_MEASURES = [
    "mw_freebase",
    "heavy_atoms",
    "aromatic_rings",
    "rtb",
    "num_ro5_violations",
    "psa",
    "hba",
    "hbd",
    "qed_weighted",
    "alogp",
]
COLORCET_SEQUENTIAL_CMAP = "CET_L17"
COLORCET_DIVERGENT_CMAP = "CET_D1A"
COLORCET_CATEGORICAL_PALETTE = "glasbey"
FIGURE_SPECS = {
    "small": {
        "figsize": (3, 3),
        "dpi": 300,
        "point_size": 1.5,
        "point_alpha": 0.86,
        "title_size": 8.5,
        "label_size": 8,
        "colorbar_label_size": 5.5,
        "tick_label_size": 5,
        "legend_fontsize": 5.5,
        "legend_title_fontsize": 6,
        "legend_marker_size": 3.8,
        "legend_max": 8,
        "colorbar_width": "3%",
        "colorbar_height": "24%",
        "colorbar_borderpad": 0.7,
    },
    "large": {
        "figsize": (9.0, 7.5),
        "dpi": 300,
        "point_size": 3.2,
        "point_alpha": 0.84,
        "title_size": 14,
        "label_size": 11,
        "colorbar_label_size": 8,
        "tick_label_size": 7,
        "legend_fontsize": 7,
        "legend_title_fontsize": 8,
        "legend_marker_size": 5.5,
        "legend_max": 14,
        "colorbar_width": "2.2%",
        "colorbar_height": "28%",
        "colorbar_borderpad": 0.9,
    },
}
DEFAULT_FIGURE_SIZE = "large"


def _colorcet_cmap(name, fallback):
    if hasattr(cc.cm, name):
        return getattr(cc.cm, name)
    try:
        return cc.cm[name]
    except (KeyError, TypeError):
        return plt.get_cmap(fallback)


def _colorcet_palette(name, n_colors):
    palette = getattr(cc, name, None)
    if palette is None:
        palette = getattr(cc, "palette", {}).get(name)
    if palette is None:
        cmap = plt.get_cmap("tab20", max(n_colors, 1))
        return [cmap(i / max(n_colors - 1, 1)) for i in range(n_colors)]
    palette = list(palette)
    return [palette[i % len(palette)] for i in range(n_colors)]


def pacmap_run_id(params):
    return (
        f"nn{params['n_neighbors']:03d}_mn{_slug(params['MN_ratio'])}_fp{_slug(params['FP_ratio'])}"
    )


def plot_pacmap_measure(
    coords,
    meta,
    color_column,
    *,
    params=None,
    output_path=None,
    show=True,
    plot_size=DEFAULT_FIGURE_SIZE,
    legend_max=None,
    legend_loc="lower right",
    legend_bbox=(0.98, 0.02),
    sequential_measures=SEQUENTIAL_MEASURES,
    sequential_cmap=COLORCET_SEQUENTIAL_CMAP,
    divergent_cmap=COLORCET_DIVERGENT_CMAP,
    categorical_palette=COLORCET_CATEGORICAL_PALETTE,
):
    if color_column not in meta.columns:
        raise ValueError(f"{color_column!r} not in metadata. Available: {meta.columns.tolist()}")

    color_series = meta[color_column]
    is_numeric = pd.api.types.is_numeric_dtype(color_series) and not pd.api.types.is_bool_dtype(
        color_series
    )
    col_label = COLUMN_LABELS.get(color_column, color_column) or str(color_column)
    valid_coords = np.isfinite(coords).all(axis=1)

    if plot_size not in FIGURE_SPECS:
        raise ValueError(f"Unknown plot_size {plot_size!r}. Use one of {list(FIGURE_SPECS)}")
    figure_spec = FIGURE_SPECS[plot_size]
    legend_max = figure_spec["legend_max"] if legend_max is None else legend_max

    fig, ax = plt.subplots(figsize=figure_spec["figsize"])

    if is_numeric:
        numeric_values = pd.to_numeric(color_series, errors="coerce")
        mask = valid_coords & numeric_values.notna().to_numpy()
        if not mask.any():
            raise ValueError(f"No finite values available for {color_column!r}")
        cmap_name = sequential_cmap if color_column in sequential_measures else divergent_cmap
        scatter = ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=numeric_values.to_numpy()[mask],
            s=figure_spec["point_size"],
            alpha=figure_spec["point_alpha"],
            cmap=_colorcet_cmap(cmap_name, "viridis").reversed(),
            linewidths=0,
            rasterized=True,
        )
        cax = inset_axes(
            ax,
            width=figure_spec["colorbar_width"],
            height=figure_spec["colorbar_height"],
            loc="lower right",
            borderpad=figure_spec["colorbar_borderpad"],
        )
        cb = fig.colorbar(scatter, cax=cax)
        cb.ax.set_title(str(col_label), fontsize=figure_spec["colorbar_label_size"], pad=3)
        cb.ax.tick_params(labelsize=figure_spec["tick_label_size"])
    else:
        mask = valid_coords & color_series.notna().to_numpy()
        if not mask.any():
            raise ValueError(f"No finite values available for {color_column!r}")
        categories = color_series[mask].astype(str)
        unique_cats = sorted(pd.unique(categories))
        palette = _colorcet_palette(categorical_palette, len(unique_cats))
        cat_to_int = {c: i for i, c in enumerate(unique_cats)}
        point_colors = [palette[cat_to_int[c]] for c in categories]
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=point_colors,
            s=figure_spec["point_size"],
            alpha=figure_spec["point_alpha"],
            linewidths=0,
            rasterized=True,
        )
        handles = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=palette[i],
                markersize=figure_spec["legend_marker_size"],
                label=str(c),
            )
            for i, c in enumerate(unique_cats[:legend_max])
        ]
        ax.legend(
            handles=handles,
            title=str(col_label),
            bbox_to_anchor=legend_bbox,
            loc=legend_loc,
            fontsize=figure_spec["legend_fontsize"],
            title_fontsize=figure_spec["legend_title_fontsize"],
            frameon=True,
            framealpha=0.72,
            borderpad=0.25,
            labelspacing=0.25,
            handlelength=0.8,
        )

    title = f"PaCMAP - {col_label}"
    ax.set_title(title, fontsize=figure_spec["title_size"], pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    fig.tight_layout()
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=figure_spec["dpi"], bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


plot_pacmap_measure(
    coords,
    meta,
    COLOR_COLUMN,
    params={
        "n_neighbors": 30,
        "MN_ratio": 0.0,
        "FP_ratio": 12.0,
    },
    plot_size="large",
)

## Sweep PaCMAP parameters

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sweep_records = []
for run_index, params in enumerate(PACMAP_PARAMETER_GRID, start=1):
    run_id = pacmap_run_id(params)
    print(f"[{run_index:02d}/{len(PACMAP_PARAMETER_GRID)}] Running {run_id}: {params}")
    reducer = pacmap.PaCMAP(n_components=2, verbose=False, **params)
    sweep_coords = reducer.fit_transform(embeddings, init="pca")

    for measure in available_measures:
        for plot_size in FIGURE_SPECS:
            output_path = FIGURE_DIR / plot_size / f"{run_id}_{_slug(measure)}_{plot_size}.png"
            plot_pacmap_measure(
                sweep_coords,
                meta,
                measure,
                params=params,
                output_path=output_path,
                show=False,
                plot_size=plot_size,
            )
            sweep_records.append(
                {
                    "run_id": run_id,
                    "measure": measure,
                    "plot_size": plot_size,
                    "figure_path": str(output_path),
                    **params,
                }
            )

manifest = pd.DataFrame(sweep_records)
manifest_path = FIGURE_DIR / "pacmap_sweep_manifest.csv"
manifest.to_csv(manifest_path, index=False)

config_path = FIGURE_DIR / "pacmap_sweep_config.json"
config_path.write_text(
    json.dumps(
        {
            "embedding_dir": str(EMBED_DIR),
            "source_parquet": str(SOURCE_PARQUET),
            "figure_dir": str(FIGURE_DIR),
            "n_parameter_combinations": len(PACMAP_PARAMETER_GRID),
            "parameter_grid": PACMAP_PARAMETER_GRID,
            "measures": available_measures,
            "missing_measures": missing_measures,
            "figure_specs": FIGURE_SPECS,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Wrote {len(manifest)} figures to {FIGURE_DIR}")
print(f"Wrote manifest: {manifest_path}")
print(f"Wrote config: {config_path}")